# 📊 CÔNG CỤ TỰ ĐỘNG LẬP BỘ BÁO CÁO TÀI CHÍNH (THÔNG TƯ 200/2014/TT-BTC)
**Tác giả:** [tuna1710](https://github.com/tuna1710/financial-statement-tool)  
**Mục tiêu:** Tự động đọc Bảng cân đối tài khoản (Trial Balance) các quý (Q1, Q2, Q3, Q4...) và xuất Bộ Báo cáo Tài chính hoàn chỉnh (B01-DN, B02-DN, B03-DN, KPI Dashboard).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tuna1710/financial-statement-tool/blob/main/Bao_Cao_Tai_Chinh_Colab.ipynb)

## 🚀 BƯỚC 1: Cài đặt môi trường & Tải mã nguồn từ GitHub

In [ ]:
# Clone repository từ GitHub và cài đặt thư viện phụ thuộc
import os, shutil

if not os.path.exists("financial-statement-tool"):
    !git clone https://github.com/tuna1710/financial-statement-tool.git
    %cd financial-statement-tool
else:
    %cd financial-statement-tool
    !git pull

!pip install -q openpyxl pandas matplotlib seaborn

## 📂 BƯỚC 2: Chọn phương thức nạp dữ liệu
Bạn có thể chọn **1 trong 3 cách** dưới đây:
- **Cách A:** Sử dụng dữ liệu mẫu có sẵn trong thư mục 
- **Cách B:** Tải file Excel trực tiếp từ máy tính lên Colab (chạy ô bên dưới)
- **Cách C:** Kết nối với Google Drive của bạn

In [ ]:
# (Tùy chọn) Cách B: Tải các file Bảng cân đối tài khoản từ máy tính của bạn
from google.colab import files

upload_custom = False # Đổi thành True nếu bạn muốn upload file mới từ máy tính

if upload_custom:
    print("Vui lòng chọn các file Excel Bảng cân đối tài khoản (.xlsx) từ máy tính:")
    uploaded = files.upload()
    os.makedirs("custom_data", exist_ok=True)
    for fn in uploaded.keys():
        shutil.move(fn, os.path.join("custom_data", fn))
    DATA_DIR = "custom_data"
    print(f"Đã tải lên {len(uploaded)} file vào thư mục custom_data!")
else:
    DATA_DIR = "data"
    print("Sử dụng dữ liệu mẫu trong thư mục data/")

## ⚡ BƯỚC 3: Tự động xử lý & Lập Báo cáo Tài chính

In [ ]:
# Chạy công cụ tự động lập báo cáo tài chính đa kỳ
OUTPUT_FILE = "output/Bo_Bao_Cao_Tai_Chinh_Tong_Hop.xlsx"

!python lap_bao_cao_tai_chinh.py --dir {DATA_DIR} --output {OUTPUT_FILE}

## 📈 BƯỚC 4: Biểu đồ Dashboard phân tích tài chính trực quan

In [ ]:
import openpyxl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập style đồ họa đẹp mắt
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "sans-serif"

# Đọc dữ liệu từ sheet Dashboard
wb = openpyxl.load_workbook(OUTPUT_FILE, data_only=True)
ws1 = wb["Dashboard & Phân tích"]

# Lấy danh sách cột các kỳ
headers = [ws1.cell(6, c).value for c in range(1, ws1.max_column + 1) if ws1.cell(6, c).value]
period_cols = [c for c in headers if "Quý" in str(c)]

metrics = {}
for r in range(7, 20):
    row_name = ws1.cell(r, 1).value
    if row_name:
        row_vals = [ws1.cell(r, c).value for c in range(3, 3 + len(period_cols))]
        metrics[row_name.strip()] = row_vals

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Biểu đồ Doanh thu & Lợi nhuận
ax1 = axes[0, 0]
x = range(len(period_cols))
revs = [v / 1e9 for v in metrics.get("Doanh thu thuần", [])]
profits = [v / 1e9 for v in metrics.get("Lợi nhuận sau thuế (EAT)", [])]
width = 0.35
ax1.bar([i - width/2 for i in x], revs, width, label="Doanh thu thuần", color="#1B365D")
ax1.bar([i + width/2 for i in x], profits, width, label="Lợi nhuận sau thuế", color="#2E7D32")
ax1.set_xticks(x)
ax1.set_xticklabels(period_cols, fontsize=11, fontweight="bold")
ax1.set_ylabel("Tỷ VNĐ", fontsize=11)
ax1.set_title("Diễn biến Doanh thu & Lợi nhuận sau thuế qua các quý", fontsize=13, fontweight="bold", pad=12)
ax1.legend(frameon=True)

# 2. Biên lợi nhuận (%)
ax2 = axes[0, 1]
gm = [v * 100 for v in metrics.get("Tỷ suất Lợi nhuận gộp (Gross Margin)", [])]
nm = [v * 100 for v in metrics.get("Tỷ suất Lợi nhuận ròng (Net Margin)", [])]
ax2.plot(period_cols, gm, marker="o", linewidth=2.5, markersize=8, label="Biên LN gộp (%)", color="#C0392B")
ax2.plot(period_cols, nm, marker="s", linewidth=2.5, markersize=8, label="Biên LN ròng (%)", color="#2980B9")
for i, txt in enumerate(gm): ax2.annotate(f"{txt:.1f}%", (i, gm[i] + 1), fontweight="bold", color="#C0392B")
for i, txt in enumerate(nm): ax2.annotate(f"{txt:.1f}%", (i, nm[i] + 1), fontweight="bold", color="#2980B9")
ax2.set_ylabel("%", fontsize=11)
ax2.set_title("Biên Lợi nhuận gộp & Biên Lợi nhuận ròng", fontsize=13, fontweight="bold", pad=12)
ax2.legend(frameon=True)

# 3. Tổng tài sản vs Vốn chủ sở hữu
ax3 = axes[1, 0]
assets = [v / 1e9 for v in metrics.get("Tổng tài sản cuối kỳ", [])]
equity = [v / 1e9 for v in metrics.get("Vốn chủ sở hữu cuối kỳ", [])]
ax3.plot(period_cols, assets, marker="D", linewidth=2.5, markersize=8, label="Tổng tài sản", color="#8E44AD")
ax3.plot(period_cols, equity, marker="^", linewidth=2.5, markersize=8, label="Vốn chủ sở hữu", color="#27AE60")
for i, txt in enumerate(assets): ax3.annotate(f"{txt:.1f} tỷ", (i, assets[i] + 5), fontweight="bold")
for i, txt in enumerate(equity): ax3.annotate(f"{txt:.1f} tỷ", (i, equity[i] + 5), fontweight="bold")
ax3.set_ylabel("Tỷ VNĐ", fontsize=11)
ax3.set_title("Quy mô Tổng tài sản & Vốn chủ sở hữu", fontsize=13, fontweight="bold", pad=12)
ax3.legend(frameon=True)

# 4. Tỷ số thanh toán hiện hành & thanh toán tức thời
ax4 = axes[1, 1]
cr = metrics.get("Tỷ số thanh toán hiện hành (CR)", [])
cash_r = metrics.get("Tỷ số thanh toán tức thời (Cash Ratio)", [])
ax4.bar([i - width/2 for i in x], cr, width, label="Thanh toán hiện hành (CR)", color="#F39C12")
ax4.bar([i + width/2 for i in x], cash_r, width, label="Thanh toán tiền mặt (Cash Ratio)", color="#16A085")
ax4.axhline(1.0, color="red", linestyle="--", alpha=0.7, label="Ngưỡng an toàn (1.0x)")
ax4.set_xticks(x)
ax4.set_xticklabels(period_cols, fontsize=11, fontweight="bold")
ax4.set_ylabel("Lần", fontsize=11)
ax4.set_title("Chỉ số khả năng thanh toán", fontsize=13, fontweight="bold", pad=12)
ax4.legend(frameon=True)

plt.tight_layout()
plt.show()

## 📥 BƯỚC 5: Tải file Báo cáo Excel kết quả về máy tính

In [ ]:
# Tải file Excel báo cáo hoàn chỉnh về máy tính của bạn
from google.colab import files

print(f"Chuẩn bị tải file: {OUTPUT_FILE}")
files.download(OUTPUT_FILE)